In [16]:
import os
import json
from dotenv import load_dotenv
import anthropic
import chromadb
from chromadb.utils import embedding_functions
import bleach

load_dotenv("dppbot/.env")
api_key = os.getenv("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=api_key)

print("✅ All libraries imported!")
print("✅ API key loaded!")

✅ All libraries imported!
✅ API key loaded!


In [17]:
chroma_client = chromadb.PersistentClient(path="dppbot/chromadb_store")

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

print("✅ ChromaDB initialized!")
print("⏳ Note: First run downloads 80MB model — wait if it takes 2-3 minutes...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\JUST BUY PC\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JUST BUY PC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ChromaDB initialized!
⏳ Note: First run downloads 80MB model — wait if it takes 2-3 minutes...


In [18]:
def chunk_text(text, min_words=30, max_words=300):
    paragraphs = text.split('\n\n')
    chunks = []
    for para in paragraphs:
        words = para.split()
        if min_words <= len(words) <= max_words:
            chunks.append(para.strip())
    return chunks

collections = {}
reg_folder = "dppbot/data/regulations"
total_chunks = 0

for filename in os.listdir(reg_folder):
    if filename.endswith(".txt"):
        collection_name = filename.replace(".txt", "").replace("_", "-")
        
        collection = chroma_client.get_or_create_collection(
            name=collection_name,
            embedding_function=embedding_function
        )
        
        with open(f"{reg_folder}/{filename}", "r") as f:
            text = f.read()
        
        chunks = chunk_text(text)
        
        if chunks:
            collection.add(
                documents=chunks,
                ids=[f"{collection_name}-{i}" for i in range(len(chunks))]
            )
        
        collections[collection_name] = collection
        total_chunks += len(chunks)
        print(f"✅ {filename} — {len(chunks)} chunks loaded")

print(f"\n✅ Total chunks in ChromaDB: {total_chunks}")

✅ certifications_guide.txt — 2 chunks loaded
✅ espr_regulation.txt — 3 chunks loaded
✅ pakistan_exporter_guide.txt — 4 chunks loaded
✅ product_specific.txt — 1 chunks loaded
✅ reach_requirements.txt — 2 chunks loaded
✅ zdhc_guide.txt — 1 chunks loaded

✅ Total chunks in ChromaDB: 13


In [19]:
def search_knowledge_base(query, n_results=3):
    clean_query = bleach.clean(query, strip=True)[:500]
    all_results = []
    
    for name, collection in collections.items():
        results = collection.query(
            query_texts=[clean_query],
            n_results=min(n_results, collection.count())
        )
        if results['documents'][0]:
            all_results.extend(results['documents'][0])
    
    return all_results

def generate_answer(query, context_chunks):
    clean_query = bleach.clean(query, strip=True)[:500]
    context = "\n\n".join(context_chunks)
    
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""You are DPPBot, an EU Digital Product Passport compliance expert for Pakistani textile exporters.

Use this context to answer the question:
{context}

Question: {clean_query}

Give a clear, practical answer focused on Pakistani exporters."""
        }]
    )
    
    return message.content[0].text

print("✅ Search and answer functions ready!")

# Test it
test_query = "What certifications do I need for EU textile exports?"
chunks = search_knowledge_base(test_query)
answer = generate_answer(test_query, chunks)
print(f"\nTest Question: {test_query}")
print(f"\nAnswer: {answer}")

✅ Search and answer functions ready!

Test Question: What certifications do I need for EU textile exports?

Answer: # EU Textile Export Certifications for Pakistani Exporters

Based on current EU requirements and 2027 Digital Product Passport mandates, here's what you **must have**:

## **PRIORITY 1: Essential Now (2025)**

### **OEKO-TEX Standard 100** ⭐ START HERE
- **Why:** EU buyers demand this FIRST
- **Tests:** Harmful substances in finished products
- **Cost:** PKR 80,000–200,000
- **Timeline:** 6–8 weeks
- **Validity:** 1 year
- **Action:** Contact SGS Pakistan (Karachi) or Bureau Veritas to start testing

### **REACH Compliance Testing**
- **Why:** EU chemical regulation mandatory
- **Tests:** Restricted substances declaration
- **Cost:** PKR 30,000–80,000
- **Action:** Include with OEKO-TEX testing

---

## **PRIORITY 2: Industry-Specific (Depends on Your Product)**

### **If you export ORGANIC COTTON:**
- **GOTS Certification** required
- Cost: PKR 150,000–300,000
- Timeline

In [20]:
db_info = {
    "collections": list(collections.keys()),
    "total_chunks": total_chunks,
    "embedding_model": "all-MiniLM-L6-v2",
    "chromadb_path": "dppbot/chromadb_store"
}

with open("dppbot/data/db_info.json", "w") as f:
    json.dump(db_info, f, indent=2)

print("✅ Database info saved!")
print(f"Collections: {list(collections.keys())}")
print("\n🎉 Notebook 2 COMPLETE — Ready for Notebook 3!")

✅ Database info saved!
Collections: ['certifications-guide', 'espr-regulation', 'pakistan-exporter-guide', 'product-specific', 'reach-requirements', 'zdhc-guide']

🎉 Notebook 2 COMPLETE — Ready for Notebook 3!
